# Neisseria gonorrhoeae AMR Prediction Analysis

This notebook integrates data processing, model training, and evaluation for predicting antibiotic resistance in *Neisseria gonorrhoeae*.

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_processing import load_data, preprocess_data, get_train_test_split
from src.models import get_models, evaluate_models_cv, tune_model, get_param_grids
from src.evaluation import plot_confusion_matrices, plot_roc_curves, plot_feature_importance

# Set plot style
sns.set(style="whitegrid")

## 1. Data Loading and Preprocessing

Select the antibiotic to analyze: 'Azithromycin', 'Ciprofloxacin', or 'Cefixime'.

In [2]:
ANTIBIOTIC = 'Ciprofloxacin' # Change to 'Azithromycin' or 'Cefixime' as needed
DATA_DIR = 'DATA'

print(f"Loading data for {ANTIBIOTIC}...")
raw_data, code = load_data(ANTIBIOTIC, DATA_DIR)
print(f"Data loaded. Shape: {raw_data.shape}")

# Preprocess
target_col = f'{code}_sr'
X, y = preprocess_data(raw_data, target_col)
print(f"Preprocessing complete. Features: {X.shape[1]}, Samples: {X.shape[0]}")

# Train/Test Split
X_train, X_test, y_train, y_test = get_train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

Loading data for Ciprofloxacin...
Data loaded. Shape: (3786, 8903)


/home/ekramah/BioinfoComputing_Project/N.Gonorrhoeae-AMR-prediction-ML/src/data_processing.py:74: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[col] = data[col].fillna('Unknown')


Preprocessing complete. Features: 8939, Samples: 3088
Training set: 2470 samples
Test set: 618 samples


## 2. Model Training and Cross-Validation

We will evaluate Logistic Regression, Random Forest, SVM, and Gradient Boosting using 5-fold Stratified Cross-Validation.

In [3]:
models = get_models()
cv_results = evaluate_models_cv(models, X_train, y_train)
print("Cross-Validation Results:")
display(cv_results)

Training Logistic Regression...
Training Random Forest...
Training SVM...
Training XGBoost...


/mnt/E42C87742C874092/anaconda3/lib/python3.13/site-packages/xgboost/training.py:199: UserWarning: [23:49:37] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/mnt/E42C87742C874092/anaconda3/lib/python3.13/site-packages/xgboost/training.py:199: UserWarning: [23:49:43] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/mnt/E42C87742C874092/anaconda3/lib/python3.13/site-packages/xgboost/training.py:199: UserWarning: [23:49:48] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/mnt/E42C87742C874092/anaconda3/lib/python3.13/site-packages/xgboost/training.py:199: UserWarning: [23:49:53] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/mnt/E42C87742C8

Training CatBoost...
Cross-Validation Results:


,Accuracy,Precision,Recall,F1 Score
Logistic Regression,0.961538,0.953371,0.964104,0.958653
Random Forest,0.963158,0.959868,0.960599,0.960167
SVM,0.955061,0.945699,0.957975,0.951731
XGBoost,0.966397,0.957143,0.971114,0.963955
CatBoost,0.968826,0.958062,0.975504,0.966612


## 3. Hyperparameter Tuning (Optional)

Uncomment the code below to run hyperparameter tuning. This may take some time.

In [ ]:
param_grids = get_param_grids()
best_models = {}

for name, model in models.items():
    print(f"Tuning {name}...")
    best_model, best_params = tune_model(model, param_grids[name], X_train, y_train)
    best_models[name] = best_model
    print(f"Best params for {name}: {best_params}")
    
models = best_models # Update models with tuned versions

Tuning Logistic Regression...


## 4. Final Evaluation on Test Set

Train the models on the full training set and evaluate on the held-out test set.

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

print("Test Set Evaluation:")
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"\n{name} Accuracy: {acc:.4f}")
    print(classification_report(y_test, y_pred))

## 5. Visualizations

### Confusion Matrices

In [ ]:
fig_cm = plot_confusion_matrices(models, X_test, y_test)
plt.show()

### ROC Curves

In [ ]:
fig_roc = plot_roc_curves(models, X_test, y_test)
plt.show()

## 6. Feature Importance and Biological Interpretation

We extract the most important features (unitigs) from the Random Forest model.

In [ ]:
rf_model = models['Random Forest']
feature_names = X.columns

fig_imp = plot_feature_importance(rf_model, feature_names, top_n=20, title="Top 20 Features (Random Forest)")
plt.show()

### Biological Mapping

The top features listed above are unitigs. To interpret them biologically, you would map these sequences to known AMR genes using BLAST or a reference database. 

For example, you can take the unitig sequence (which is the column name or can be retrieved if column names are IDs) and search against the CARD database.